# DataFrames de panel PISA y deberes

Se combinan las medias de las asignaturas con las variables de deberes por estrato y por pais.

In [1]:
from pathlib import Path
import pandas as pd
import gc
import sys
import os

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))


from src.utils_pisa import *


%load_ext autoreload
%autoreload 2


ruta_datos = Path('../../data/processed')

establecer_semilla(42)
pd.set_option('display.float_format', lambda valor: f'{valor:.2f}')


✅ Semillas del proyecto establecidas con éxito (seed=42).


In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## Panel por pais y estrato

La union usa `CNT`, `STRATUM` y `year` para asociar las observaciones del mismo pais, estrato y edicion.

In [2]:
df_medias_estrato = pd.read_parquet(
    ruta_datos / 'pisa_medias_asignaturas_estrato_anual.parquet'
)
df_deberes_strat = pd.read_parquet(
    ruta_datos / 'deberes_strat.parquet'
)

df_panel_estrato = df_medias_estrato.merge(
    df_deberes_strat,
    on=['CNT', 'STRATUM', 'year'],
    how='inner',
    validate='one_to_one',
)
df_panel_estrato = df_panel_estrato[
    ['CNT', 'STRATUM', 'year']
    + [col for col in df_panel_estrato.columns if col not in ['CNT', 'STRATUM', 'year']]
]

display(df_panel_estrato.head())
print(f'Dimensiones del panel por estrato: {df_panel_estrato.shape}')

,CNT,STRATUM,year,media_math_estrato,media_read_estrato,media_scie_estrato,tiempo_deberes_totales_diarios,dispone_personal_apoyo
0,USA,USA08,2022,476.10,520.35,512.01,3.07,1.00
1,USA,USA07,2022,539.40,535.90,592.47,3.40,NaN
2,USA,USA06,2022,454.64,493.90,490.45,2.55,1.07
3,USA,USA05,2022,476.43,507.31,490.59,3.05,1.00
4,USA,USA04,2022,469.99,503.32,504.79,2.60,1.09


Dimensiones del panel por estrato: (2328, 8)


## Panel por pais

La union usa `CNT` y `year` para asociar las observaciones del mismo pais y edicion.

In [3]:
df_medias_pais = pd.read_parquet(
    ruta_datos / 'pisa_medias_asignaturas_pais_anual.parquet'
)
df_deberes_cnt = pd.read_parquet(
    ruta_datos / 'deberes_cnt.parquet'
)

df_panel_pais = df_medias_pais.merge(
    df_deberes_cnt,
    on=['CNT', 'year'],
    how='inner',
    validate='one_to_one',
)
df_panel_pais = df_panel_pais[
    ['CNT', 'year']
    + [col for col in df_panel_pais.columns if col not in ['CNT', 'year']]
]

display(df_panel_pais.head())
print(f'Dimensiones del panel por pais: {df_panel_pais.shape}')

,CNT,year,media_math_pais,media_read_pais,media_scie_pais,tiempo_deberes_totales_diarios,dispone_personal_apoyo
0,USA,2022,464.89,503.94,499.41,2.73,1.05
1,TUR,2022,453.15,456.08,475.94,3.14,1.62
2,SWE,2022,481.77,486.98,493.55,2.44,1.07
3,SVN,2022,484.53,468.54,499.96,2.44,1.54
4,SVK,2022,463.99,446.86,462.27,2.35,1.30


Dimensiones del panel por pais: (132, 7)


---

# AGREGACION DE LOS **DATOS ECONOMICOS SIMULADOS**

In [12]:
ruta_datos_simulados = Path('../../notebooks/fase_2_feature_engineering')
#pisa_datos_economicos_simulados

In [13]:
def ordenar_columnas_panel(df):
    claves = [col for col in ['CNT', 'STRATUM', 'year'] if col in df.columns]
    medias = [
        col for col in [
            'media_math_pais', 'media_read_pais', 'media_scie_pais',
            'media_math_estrato', 'media_read_estrato', 'media_scie_estrato',
        ]
        if col in df.columns
    ]
    resto = [col for col in df.columns if col not in claves + medias]
    return df[claves + medias + resto]


# Paneles en formato largo
df_panel_estrato = ordenar_columnas_panel(df_panel_estrato)
df_panel_estrato_final = ordenar_columnas_panel(df_panel_estrato_final)
df_panel_pais = ordenar_columnas_panel(df_panel_pais)
df_pais_transaccional = ordenar_columnas_panel(df_pais_transaccional)


# Panel por país en formato ancho: medias primero, agrupadas por asignatura
if 'df_pais_ancho' in globals():
    orden_medias = [
        col
        for prefijo in ['media_math_pais_', 'media_read_pais_', 'media_scie_pais_']
        for col in df_pais_ancho.columns
        if col.startswith(prefijo)
    ]
    claves = ['CNT'] if 'CNT' in df_pais_ancho.columns else []
    resto = [
        col for col in df_pais_ancho.columns
        if col not in claves + orden_medias
    ]
    df_pais_ancho = df_pais_ancho[claves + orden_medias + resto]

In [10]:
# ... [Tu código actual que genera df_panel_estrato y df_panel_pais] ...

# =====================================================================
# INTEGRACIÓN Y PIVOTADO DE VARIABLES ECONÓMICAS ALEATORIAS
# =====================================================================

# 1. Leemos el archivo parquet base aleatorio
df_macro_base = pd.read_parquet(ruta_datos_simulados/ 'pisa_datos_economicos_simulados.parquet')

# Aseguramos tipos uniformes para los cruces
df_macro_base['year'] = df_macro_base['year'].astype(int)
df_panel_estrato['year'] = df_panel_estrato['year'].astype(int)
df_panel_pais['year'] = df_panel_pais['year'].astype(int)


# ---------------------------------------------------------------------
# UNIÓN A: TABLA LARGA POR ESTRATOS (Formato transaccional para modelos)
# ---------------------------------------------------------------------
df_panel_estrato_final = df_panel_estrato.merge(
    df_macro_base,
    on=['CNT', 'year'],
    how='inner'
)

columnas_macro = ['PIB_per_capita_PPP', 'Gasto_Alumno_PPP', 'Gasto_Educacion_PIB', 'Indice_Gini', 'ESCS_media_pais_anual']
df_panel_estrato_final = df_panel_estrato_final[[col for col in df_panel_estrato_final.columns if col not in [['CNT', 'STRATUM', 'year']]+ columnas_macro]+ columnas_macro
]

print("=== 1. PANEL POR ESTRATO FINAL COMPLETO (TABLA LARGA) ===")
display(df_panel_estrato_final.head())
print(f'Dimensiones del panel largo: {df_panel_estrato_final.shape}\n')


=== 1. PANEL POR ESTRATO FINAL COMPLETO (TABLA LARGA) ===


,CNT,STRATUM,year,media_math_estrato,media_read_estrato,media_scie_estrato,tiempo_deberes_totales_diarios,dispone_personal_apoyo,PIB_per_capita_PPP,Gasto_Alumno_PPP,Gasto_Educacion_PIB,Indice_Gini,ESCS_media_pais_anual
0,USA,USA08,2022,476.10,520.35,512.01,3.07,1.00,24568.52,56890.87,4.63,39.15,0.21
1,USA,USA07,2022,539.40,535.90,592.47,3.40,NaN,24568.52,56890.87,4.63,39.15,0.21
2,USA,USA06,2022,454.64,493.90,490.45,2.55,1.07,24568.52,56890.87,4.63,39.15,0.21
3,USA,USA05,2022,476.43,507.31,490.59,3.05,1.00,24568.52,56890.87,4.63,39.15,0.21
4,USA,USA04,2022,469.99,503.32,504.79,2.60,1.09,24568.52,56890.87,4.63,39.15,0.21


Dimensiones del panel largo: (2328, 13)



In [11]:
# ---------------------------------------------------------------------
# UNIÓN B: TABLA POR PAÍS 
# ---------------------------------------------------------------------
# Primero unimos tus datos de df_panel_pais con la matriz macro base
df_pais_transaccional = df_panel_pais.merge(
    df_macro_base,
    on=['CNT', 'year'],
    how='inner'
)

# Identificamos todas las columnas que queremos pivotar (las tuyas originales + las nuevas económicas)
columnas_a_pivotar = [col for col in df_pais_transaccional.columns if col not in ['CNT', 'year']]

# Aplicamos tu lógica exacta: pivotamos para que los países sean filas y los años columnas
df_pais_ancho = df_pais_transaccional.pivot(
    index='CNT',
    columns='year',
    values=columnas_a_pivotar
)

# Aplanamos el nivel de las columnas para que quede limpio de leer (ej: PIB_per_capita_PPP_2022)
df_pais_ancho.columns = [f'{var}_{anio}' for var, anio in df_pais_ancho.columns]
df_pais_ancho = df_pais_ancho.reset_index()

print("=== 2. PANEL POR PAÍS FINAL COMPLETO (FORMATO ANCHO PIVOTADO) ===")
display(df_pais_ancho.head())
print(f'Dimensiones del panel ancho: {df_pais_ancho.shape}')

=== 2. PANEL POR PAÍS FINAL COMPLETO (FORMATO ANCHO PIVOTADO) ===


,CNT,media_math_pais_2012,media_math_pais_2015,media_math_pais_2018,media_math_pais_2022,media_read_pais_2012,media_read_pais_2015,media_read_pais_2018,media_read_pais_2022,media_scie_pais_2012,...,Gasto_Educacion_PIB_2018,Gasto_Educacion_PIB_2022,Indice_Gini_2012,Indice_Gini_2015,Indice_Gini_2018,Indice_Gini_2022,ESCS_media_pais_anual_2012,ESCS_media_pais_anual_2015,ESCS_media_pais_anual_2018,ESCS_media_pais_anual_2022
0,AUS,504.15,493.90,491.36,487.08,511.80,502.90,502.63,498.05,521.49,...,6.75,5.36,37.97,38.03,28.31,33.80,-1.03,0.62,-0.95,-0.63
1,AUT,505.54,496.74,498.94,487.27,489.61,484.87,484.39,480.41,505.78,...,3.29,3.44,32.16,37.81,46.72,40.11,-0.13,-1.36,1.40,-0.18
2,BEL,514.53,506.98,508.07,489.49,508.62,498.52,492.86,478.85,504.87,...,7.23,3.88,45.73,36.67,45.37,24.13,-0.72,-0.95,0.29,-0.52
3,CAN,518.07,515.65,512.02,496.95,523.12,526.67,520.09,507.13,525.44,...,3.02,3.33,31.92,24.86,43.39,31.96,-0.66,1.46,0.62,-1.15
4,CHE,530.93,521.25,515.31,507.99,509.04,492.20,483.93,483.33,515.30,...,6.42,4.92,24.59,45.18,37.03,23.64,-0.57,-0.08,0.81,-1.18


Dimensiones del panel ancho: (33, 41)



---


In [14]:
df_panel_estrato_final.columns

Index(['CNT', 'STRATUM', 'year', 'PIB_per_capita_PPP', 'Gasto_Alumno_PPP',
       'Gasto_Educacion_PIB', 'Indice_Gini', 'ESCS_media_pais_anual',
       'media_math_estrato', 'media_read_estrato', 'media_scie_estrato',
       'tiempo_deberes_totales_diarios', 'dispone_personal_apoyo'],
      dtype='str')

# *Feature engenieering*

## Calculo de la presión académica


Para evaluar el impacto real de la carga de trabajo en casa, se crea una nueva métrica continua denominada **Presión Académica Neta**. Esta variable relaciona directamente la cantidad de deberes con los recursos de apoyo que ofrece el centro escolar, bajo la premisa de que el apoyo reduce la carga escolar

**Fórmula:**
$$Presion\_Academica\_Neta = \frac{tiempo\_deberes\_totales\_diarios}{1 + dispone\_personal\_apoyo}$$

** Denominador ($1 + apoyo$):**
 (Efecto amortiguador):** El denominador actúa como un divisor de la carga de trabajo. 
  * Si el colegio **no tiene apoyo** (0), el denominador es $1 + 0 = 1$. El alumno absorbe el 100% de la carga de los deberes.
  * Si el colegio **sí tiene apoyo** (1), el denominador es $1 + 1 = 2$. La presión neta se reduce a la mitad, reflejando cómo los profesores de refuerzo alivian el esfuerzo solitario en casa.

In [14]:
df_panel_estrato_final['Presion_Academica_Neta'] = df_panel_estrato_final['tiempo_deberes_totales_diarios'] / (1 + df_panel_estrato_final['dispone_personal_apoyo'])

In [18]:
df_panel_estrato_final['Presion_Academica_Neta'].describe()

count   1944.00
mean       1.19
std        0.52
min        0.00
25%        0.86
50%        1.11
75%        1.41
max        6.70
Name: Presion_Academica_Neta, dtype: float64

## Discretización del Nivel Socioeconómico (ECS)



Se transforma el índice continuo de estatus socioeconómico (`ECS_nivel`) en una variable categórica ordinal. De tal forma que se analice el impacto de las variables educativas a través de un indice cualitativo mas intuitivo

**Regla de codificación:**
Se utiliza una discretización basada en cuantiles para dividir los estratos en tres grupos. De esta forma se pueden aprender patrones de cada grupo social

* **Bajo:** menor índice socioeconómico.
* **Medio:** punto intermedio del cuantil.
* **Alto:** mayor índice socioeconómico.

Esta segmentación permitirá evaluar interacciones complejas, como por ejemplo, si la *Presión Académica Neta* tiene un efecto dispar en el rendimiento dependiendo de los recursos del hogar y del entorno del estrato.

In [24]:
# Etiquetas para los 3 niveles
etiquetas_socioeconomicas = ['Bajo', 'Medio', 'Alto']

# Dividimos la variable continua en 3 partes iguales ( q=3 )
df_panel_estrato_final['ECS_categoria'] = pd.qcut(
    df_panel_estrato_final['ESCS_media_pais_anual'], 
    q=3, 
    labels=etiquetas_socioeconomicas
)

# Visualizamos cómo ha quedado el reparto de estratos por cada nivel
print("Porcentaje de distribución de estratos por nivel socioeconómico:")
print(df_panel_estrato_final['ECS_categoria'].value_counts(normalize=True) * 100)

Porcentaje de distribución de estratos por nivel socioeconómico:
ECS_categoria
Medio   36.94
Bajo    33.33
Alto    29.73
Name: proportion, dtype: float64


<div style="
  background: linear-gradient(135deg, rgba(128, 128, 128, 0.25) 0%, rgba(64, 64, 64, 0.15) 100%);
  backdrop-filter: blur(4px);
  -webkit-backdrop-filter: blur(4px);
  border-left: 5px solid rgba(128, 128, 128, 0.6);
  padding: 16px;
  border-radius: 8px;
  color: inherit;
  margin: 20px 0;
">
  <strong style="display: block; margin-bottom: 6px; font-size: 1.1em;">
    💡 Division a nivel global
  </strong>
    El código anterior calcula los terciles de forma global (comparando todos los estratos de todos los países a la vez). Si en tu investigación prefieres que un estrato se considere "Alto" o "Bajo" en relación a su propio país (lo cual a veces es más preciso en estudios internacionales). Se debe aaplicar el qcut agrupando por país ('CNT')

</div>

## Perfil Macroeconómico



Para dotar al modelo de un contexto estructural simplificado, se segmentan los países en perfiles macroeconómicos utilizando un algoritmo no supervisado (K-Means). 

**Metodología:**
1. Se aísla un *dataframe* transversal a nivel de país con los indicadores estructurales: `PIB_per_capita_PPP`, `Gasto_Alumno_PPP`, `Gasto_Educacion_PIB` e `Indice_Gini`.
2. Se escalan las variables (StandardScaler) para evitar que magnitudes como el PIB dominen el cálculo de distancias sobre porcentajes como el Gini.
3. Se entrena un modelo K-Means (previa evaluación del número óptimo de clústeres mediante el método del codo o silueta, asumiendo aquí $k=3$ o $k=4$).
4. Se asigna la etiqueta resultante (`Perfil_Pais`) a la tabla longitudinal larga, mapeando cada estrato a la realidad estructural de su nación.